# Entity clustering — current character pipeline
Character-only extraction from the current full inference pipeline. Local Magi detections/page clusters → existing fine-tuned final two crop blocks → duplicate cleanup → cross-page entities. No Paddle, Gemma, contours or narration models. Run in godfather.venv. Own caches only. Model/checkpoint references and earlier identity experiments are listed in models.json; weights are reused in place.

In [9]:
import os
os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'
import torch
from pathlib import Path
import sys,json,hashlib,copy,gc
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display,clear_output,Image as DisplayImage
from io import BytesIO
from transformers import AutoModel,AutoConfig

PROJECT=Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
ROOT=PROJECT/'FINETUNING-EXPERMENTS/EntityClustering'
sys.path.insert(0,str(ROOT/'src'))
from entity_clustering_core import remove_duplicate_characters,clusterize,page_clusters
MODELS=json.loads((ROOT/'models.json').read_text(encoding='utf-8'))
MODEL_PATH=Path(MODELS['active']['base_model'])
CHECKPOINT_PATH=Path(MODELS['active']['checkpoint'])
ENTITY_THRESHOLD=MODELS['active']['entity_threshold']
ASSOCIATION_HEAD_PATH=None
SPLIT='development'
SEQUENCE_ID='seq_2fae297a18af2a57'
INPUT_IMAGES=None  # Or an ordered list of image Paths.
FORCE=False


## 1. Images, checkpoint and own caches
Existing detector/association weights and fine-tuned crop checkpoint stay unchanged. No other experiment cache is read.

In [10]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), ensure_ascii=True,
                                    indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

SPLIT=SPLIT.strip();SEQUENCE_ID=SEQUENCE_ID.strip()
PAGE_FILES=[Path(p) for p in INPUT_IMAGES] if INPUT_IMAGES is not None else sorted((PROJECT/'TaskData'/SPLIT/'images'/SEQUENCE_ID).glob('*.png'))
assert 1<=len(PAGE_FILES)<=3,'Supply one to three ordered images'
chapter_pages=[];crop_pages=[]
for path in PAGE_FILES:
    with Image.open(path) as image:
        chapter_pages.append(np.asarray(image.convert('L').convert('RGB')))
        crop_pages.append(np.asarray(image.convert('RGB')))
checkpoint=torch.load(CHECKPOINT_PATH,map_location='cpu',weights_only=True)
assert checkpoint['version']==2 and checkpoint['settings']['trained_blocks']==2
assert checkpoint['best_epoch']==MODELS['active']['epoch']
for name,expected in checkpoint['base_model_hashes'].items():assert file_hash(MODEL_PATH/name)==expected
INPUT_ID=dict(pages=[dict(path=str(p.resolve()),sha256=file_hash(p)) for p in PAGE_FILES],
    model=str(MODEL_PATH),model_files={p.name:[p.stat().st_size,p.stat().st_mtime_ns]
    for p in MODEL_PATH.iterdir() if p.is_file()},helper=file_hash(ROOT/'src/entity_clustering_core.py'))
CACHE=ROOT/'cache'/SEQUENCE_ID/digest(INPUT_ID)[:16]
OUTPUT=CACHE/'outputs';OUTPUT.mkdir(parents=True,exist_ok=True)


def cached(name,settings,compute):
    path=CACHE/(name+'.json');signature=digest(dict(inputs=INPUT_ID,settings=settings))
    payload=json.loads(path.read_text(encoding='utf-8')) if path.is_file() and not FORCE else None
    if not payload or payload.get('signature')!=signature:
        data=json_ready(compute());payload=dict(signature=signature,data=data,data_hash=digest(data))
        write_json(path,payload);print('CREATE',name)
    else:print('REUSE',name)
    assert digest(payload['data'])==payload['data_hash']
    return payload['data']
print('Images:',[p.name for p in PAGE_FILES]);print('Own cache:',CACHE)


Images: ['01.png', '02.png', '03.png']
Own cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\EntityClustering\cache\seq_2fae297a18af2a57\9c9c97a401a2c0b2


## 2. Reused Magi/crop functions
Original detections and within-page clusters are computed before patching the crop encoder. These functions are copied from the current full pipeline; no notebook is executed.

In [11]:
magi_model=None

def get_magi():
    global magi_model
    if magi_model is None:
        assert torch.cuda.is_available(), 'Torch CUDA unavailable'
        # Normal tensors allow the later checkpoint copy; only predictions use inference mode.
        with torch.inference_mode(False):
            config = AutoConfig.from_pretrained(str(MODEL_PATH), trust_remote_code=True, local_files_only=True)
            # The full checkpoint already contains the backbone: avoid timm downloading it again.
            config.detection_model_config.use_pretrained_backbone = False
            config.disable_ocr = False
            config.disable_crop_embeddings = False
            magi_model, info = AutoModel.from_pretrained(str(MODEL_PATH), config=config,
                trust_remote_code=True, local_files_only=True, output_loading_info=True)
            assert not info['missing_keys'] and not info.get('mismatched_keys'), info
            magi_model = magi_model.cuda().eval()
            magi_model.requires_grad_(False)
            if ASSOCIATION_HEAD_PATH is not None:
                association = torch.load(ASSOCIATION_HEAD_PATH, map_location='cpu', weights_only=True)
                magi_model.text_character_matching_head.load_state_dict(association['state_dict'], strict=True)
    return magi_model


def predict_magi():
    model = get_magi()
    with torch.inference_mode():
        return model.do_chapter_wide_prediction(
            chapter_pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)


def predict_embeddings():
    boxes = [result['characters'] for result in magi_raw_results]
    if not any(boxes):
        return [[] for _ in boxes]
    model = get_magi()
    last = len(model.crop_embedding_model.encoder.layer) - 1
    assert last >= 1
    prefixes = (f'crop_embedding_model.encoder.layer.{last-1}.',
                f'crop_embedding_model.encoder.layer.{last}.')
    parameters = dict(model.named_parameters())
    expected_names = {name for name in parameters if name.startswith(prefixes)}
    assert tuple(checkpoint['trainable_prefixes']) == prefixes
    assert set(checkpoint['state_dict']) == expected_names
    # Patch only the two crop blocks after original Magi prediction is complete.
    with torch.no_grad():
        for name, tensor in checkpoint['state_dict'].items():
            assert tensor.shape == parameters[name].shape
            parameters[name].copy_(tensor.to(parameters[name].device))
    model.eval()
    with torch.inference_mode():
        embeddings = model.predict_crop_embeddings(crop_pages, boxes, mask_ratio=0.0, batch_size=32)
        return [torch.nn.functional.normalize(x, p=2, dim=1) if len(x) else x for x in embeddings]


def validate_magi(results):
    assert len(results) == len(chapter_pages), 'Magi page count mismatch'
    for result in results:
        assert 'tails' in result and 'text_tail_associations' in result,'Magi must return tail detections/links'
        assert len(result['texts']) == len(result['ocr']), 'Text/OCR count mismatch'
        assert len(result['characters']) == len(result['character_cluster_labels']), 'Character/cluster count mismatch'
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']), 'Invalid text association index'
            assert 0 <= char_index < len(result['characters']), 'Invalid character association index'

## 3. Extract, clean and cluster
Fine-tuned embeddings are 768d and L2-normalized. Contained duplicate cleanup uses IoU 0.90/cosine 0.95; cross-page matching uses threshold 0.82. Entities are sequence-local E1, E2, etc.

In [12]:
try:
    magi_raw_results=cached('magi_raw',dict(stage='original_magi'),predict_magi)
    validate_magi(magi_raw_results)
    embeddings=cached('embeddings',dict(raw=digest(magi_raw_results),checkpoint=file_hash(CHECKPOINT_PATH)),predict_embeddings)
finally:
    magi_model=None;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
cleanup=cached('character_cleanup',dict(raw=digest(magi_raw_results),embeddings=digest(embeddings),iou=.9,cosine=.95),
    lambda:remove_duplicate_characters(magi_raw_results,embeddings,.9,.95))
character_results=cleanup['pages'];character_embeddings=cleanup['embeddings']
entities=cached('entities',dict(cleanup=digest(cleanup),threshold=ENTITY_THRESHOLD),
    lambda:clusterize(character_results,character_embeddings,ENTITY_THRESHOLD))
write_json(OUTPUT/'entities.json',entities)
write_json(OUTPUT/'character_cleanup.json',cleanup['audit'])
write_json(OUTPUT/'models.json',MODELS)
write_json(OUTPUT/'run_metadata.json',dict(inputs=INPUT_ID,checkpoint=str(CHECKPOINT_PATH),
    checkpoint_sha256=file_hash(CHECKPOINT_PATH),threshold=ENTITY_THRESHOLD,source=MODELS['provenance']))
display(pd.DataFrame(entities['decisions']))
print('Entities:',len(entities['entities']),'| Exports:',OUTPUT)


REUSE magi_raw
REUSE embeddings
REUSE character_cleanup
REUSE entities


,page_index,cluster_id,entity_id,reason,score,best_pair
0,0,PG1:CL1,E1,new_entity,NaN,None
1,0,PG1:CL2,E2,new_entity,NaN,None
2,0,PG1:CL3,E3,new_entity,NaN,None
3,0,PG1:CL4,E4,new_entity,NaN,None
4,1,PG2:CL1,E5,new_entity,NaN,None
5,1,PG2:CL2,E3,matched,0.948633,"{'current': 'PG2:C4', 'existing': 'PG1:C6'}"
6,1,PG2:CL3,E6,new_entity,NaN,None
7,1,PG2:CL4,E7,new_entity,NaN,None
8,2,PG3:CL1,E8,new_entity,NaN,None
9,2,PG3:CL2,E2,matched,0.860615,"{'current': 'PG3:C3', 'existing': 'PG1:C2'}"


Entities: 9 | Exports: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\EntityClustering\cache\seq_2fae297a18af2a57\9c9c97a401a2c0b2\outputs


## 4. Entity viewer
One selected page; stable entity colors, character IDs and optional lines connecting the same entity. No models run when changing widgets.

In [13]:
def show_entities(page_index=0,same_entity=True,ids=True):
    result=character_results[page_index]
    palette=plt.get_cmap('hsv',max(2,len(entities['entities'])+1))
    colors={eid:palette(i) for i,eid in enumerate(entities['entities'])}
    fig,ax=plt.subplots(figsize=(11,16));ax.imshow(crop_pages[page_index]);members={}
    for index,box in enumerate(result['characters']):
        cid=f'PG{page_index+1}:C{index+1}';eid=entities['char_to_entity'][cid]
        a,b,c,d=box;color=colors[eid]
        ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=2))
        ax.text(a,b,(cid+' / ' if ids else '')+eid,fontsize=8,bbox=dict(facecolor=color,alpha=.8))
        members.setdefault(eid,[]).append(((a+c)/2,(b+d)/2))
    if same_entity:
        for eid,points in members.items():
            for point in points[1:]:ax.plot([points[0][0],point[0]],[points[0][1],point[1]],color=colors[eid])
    ax.set_title(SEQUENCE_ID+' | '+PAGE_FILES[page_index].name);ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


# Use the complete page viewer at the end; this function is available for manual calls.


In [14]:
def show_entities(page_index=0,same_entity=True,ids=True):
    result=character_results[page_index]
    palette=plt.get_cmap('hsv',max(2,len(entities['entities'])+1))
    colors={eid:palette(i) for i,eid in enumerate(entities['entities'])}
    fig,ax=plt.subplots(figsize=(11,16));ax.imshow(crop_pages[page_index]);members={}
    for index,box in enumerate(result['characters']):
        cid=f'PG{page_index+1}:C{index+1}';eid=entities['char_to_entity'][cid]
        a,b,c,d=box;color=colors[eid]
        ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=2))
        ax.text(a,b,(cid+' / ' if ids else '')+eid,fontsize=8,bbox=dict(facecolor=color,alpha=.8))
        members.setdefault(eid,[]).append(((a+c)/2,(b+d)/2))
    if same_entity:
        for eid,points in members.items():
            for point in points[1:]:ax.plot([points[0][0],point[0]],[points[0][1],point[1]],color=colors[eid])
    ax.set_title(SEQUENCE_ID+' | '+PAGE_FILES[page_index].name);ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


# Use the complete page viewer at the end; this function is available for manual calls.


## 5. Text–tail versus text–character direction audit
Compare **text-center → tail-center** with **text-center → associated-character-center**. This is a bounding-box center proxy, not a tail-tip/base direction estimate. Use the cleaned character associations and original unchanged text/tail indices. Each explicit text/character/tail combination gets its own record.

Angle = arccos(cosine similarity), 0° aligned / 90° perpendicular / 180° opposite. Flag angles strictly above the adjustable threshold (default 90°). Also report angle/180 and character-distance/tail-distance ratio; the flag uses angle only. Missing tails and zero-length vectors are unassessed, not disagreements. Associations and entity assignments are never changed.

In [15]:
DIRECTION_FLAG_DEGREES=90.0  # Diagnostic threshold; no association is removed.
assert 0<=DIRECTION_FLAG_DEGREES<=180


def tail_character_geometry(text_box,tail_box,character_box,threshold):
    def midpoint(box):
        a,b,c,d=np.asarray(box,dtype=float)
        return np.array([(a+c)/2,(b+d)/2])
    origin=midpoint(text_box)
    tail_vector=midpoint(tail_box)-origin
    character_vector=midpoint(character_box)-origin
    tail_distance=float(np.linalg.norm(tail_vector))
    character_distance=float(np.linalg.norm(character_vector))
    base=dict(text_center=origin.tolist(),tail_center=midpoint(tail_box).tolist(),
        character_center=midpoint(character_box).tolist(),text_to_tail_vector=tail_vector.tolist(),
        text_to_character_vector=character_vector.tolist(),tail_distance_px=tail_distance,
        character_distance_px=character_distance)
    if not np.isfinite(np.concatenate([origin,tail_vector,character_vector])).all() or min(tail_distance,character_distance)<=1e-8:
        return dict(base,assessed=False,cosine_similarity=None,angle_degrees=None,
            angle_fraction=None,character_to_tail_distance_ratio=None,direction_flag=False,
            reason='nonfinite coordinates or zero-length center vector')
    cosine=float(np.clip(np.dot(tail_vector,character_vector)/(tail_distance*character_distance),-1,1))
    angle=float(np.degrees(np.arccos(cosine)))
    return dict(base,assessed=True,cosine_similarity=cosine,angle_degrees=angle,
        angle_fraction=angle/180,character_to_tail_distance_ratio=character_distance/tail_distance,
        direction_flag=bool(angle>threshold),reason='angle above threshold' if angle>threshold else 'angle within threshold')


direction_records=[]
for page_index,result in enumerate(character_results):
    tail_by_text={}
    for text_index,tail_index in result.get('text_tail_associations',[]):
        assert 0<=text_index<len(result['texts']) and 0<=tail_index<len(result['tails'])
        tail_by_text.setdefault(text_index,[]).append(tail_index)
    for text_index,character_index in result['text_character_associations']:
        cid=f'PG{page_index+1}:C{character_index+1}'
        base=dict(page_index=page_index,page_file=str(PAGE_FILES[page_index]),
            box_id=f'PG{page_index+1}:T{text_index+1}',text_index=text_index,
            character_index=character_index,character_id=cid,entity_id=entities['char_to_entity'][cid],
            text=result['ocr'][text_index],threshold_degrees=DIRECTION_FLAG_DEGREES)
        tail_ids=sorted(set(tail_by_text.get(text_index,[])))
        if not tail_ids:
            direction_records.append(dict(base,tail_index=None,assessed=False,cosine_similarity=None,
                angle_degrees=None,angle_fraction=None,character_to_tail_distance_ratio=None,
                direction_flag=False,reason='no explicit text-to-tail association'))
        for tail_index in tail_ids:
            metrics=tail_character_geometry(result['texts'][text_index],result['tails'][tail_index],
                result['characters'][character_index],DIRECTION_FLAG_DEGREES)
            direction_records.append(dict(base,tail_index=tail_index,**metrics))
DIRECTION_COLUMNS=['box_id','text','character_id','entity_id','tail_index','assessed',
    'cosine_similarity','angle_degrees','angle_fraction','character_to_tail_distance_ratio','direction_flag','reason']
direction_audit=pd.DataFrame(direction_records,columns=list(dict.fromkeys(DIRECTION_COLUMNS+
    [key for row in direction_records for key in row])))
flagged_directions=direction_audit.loc[direction_audit['direction_flag']==True]
display(direction_audit[DIRECTION_COLUMNS])
print('Flagged:',len(flagged_directions),'| All association/tail rows:',len(direction_audit))
write_json(OUTPUT/'tail_character_direction_audit.json',direction_records)
direction_audit.to_csv(OUTPUT/'tail_character_direction_audit.csv',index=False)
flagged_directions.to_csv(OUTPUT/'flagged_tail_character_directions.csv',index=False)


,box_id,text,character_id,entity_id,tail_index,assessed,cosine_similarity,angle_degrees,angle_fraction,character_to_tail_distance_ratio,direction_flag,reason
0,PG1:T1,"MANGA STREAN, DO NOTMIROR",PG1:C6,E3,NaN,False,NaN,NaN,NaN,NaN,False,no explicit text-to-tail association
1,PG1:T2,Got your tomato devil right here.,PG1:C6,E3,3.0,True,0.776876,39.024529,0.216803,1.606705,False,angle within threshold
2,PG1:T3,"It revolves from its seeds, so you should burn...",PG1:C6,E3,0.0,True,0.889938,27.134499,0.150747,1.717923,False,angle within threshold
3,PG1:T4,"Good work, Denji.",PG1:C1,E1,4.0,True,0.966798,14.805707,0.082254,1.460783,False,angle within threshold
4,PG1:T5,This guy's corpse'll fetch a pretty penny on t...,PG1:C1,E1,4.0,True,0.842604,32.583846,0.181021,1.976961,False,angle within threshold
5,PG1:T6,The reward's 400 grand.,PG1:C1,E1,1.0,True,0.908265,24.733325,0.137407,2.178664,False,angle within threshold
6,PG1:T7,Though you look at the bottom-of-the-the-the-a...,PG1:C1,E1,NaN,False,NaN,NaN,NaN,NaN,False,no explicit text-to-tail association
7,PG1:T8,Thanks !!,PG1:C4,E3,5.0,True,0.912406,24.160070,0.134223,1.132207,False,angle within threshold
8,PG1:T11,only 70 grand left...,PG1:C5,E3,2.0,True,0.994852,5.816038,0.032311,1.453240,False,angle within threshold
9,PG2:T1,After paying the water bill,PG2:C3,E3,0.0,True,0.987151,9.194723,0.051082,1.670309,False,angle within threshold


Flagged: 0 | All association/tail rows: 26


## 6. Direction-disagreement viewer
One selected page. Magenta arrow = text→tail center; cyan arrow = text→character center (red if flagged). Toggle flagged-only to inspect all assessable links. The table shows angle/cosine and distance ratio. Viewer performs no model inference.

In [16]:
def show_direction_audit(page_index=0,flagged_only=True,ocr=True):
    records=[row for row in direction_records if row['page_index']==page_index and row['assessed']
        and (not flagged_only or row['direction_flag'])]
    result=character_results[page_index]
    fig,ax=plt.subplots(figsize=(11,16));ax.imshow(crop_pages[page_index])
    drawn=set()
    for row in records:
        text_index=row['text_index'];character_index=row['character_index'];tail_index=row['tail_index']
        for kind,index,box,color in [('text',text_index,result['texts'][text_index],'gold'),
            ('character',character_index,result['characters'][character_index],'deepskyblue'),
            ('tail',tail_index,result['tails'][tail_index],'magenta')]:
            if (kind,index) in drawn:continue
            drawn.add((kind,index));a,b,c,d=box
            ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linewidth=2))
        origin=row['text_center']
        for endpoint,color in [(row['tail_center'],'magenta'),
            (row['character_center'],'red' if row['direction_flag'] else 'cyan')]:
            ax.annotate('',xy=endpoint,xytext=origin,arrowprops=dict(arrowstyle='->',color=color,lw=2))
        label=f"{row['box_id']} -> {row['character_id']} / {row['entity_id']} | {row['angle_degrees']:.1f} deg"
        if ocr:label+=' | '+row['text'][:70]
        ax.text(origin[0],origin[1],label,fontsize=8,bbox=dict(facecolor='salmon' if row['direction_flag'] else 'white',alpha=.85))
    ax.set_title(f'{PAGE_FILES[page_index].name} | threshold > {DIRECTION_FLAG_DEGREES:g} deg | {len(records)} links')
    ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))
    display(pd.DataFrame(records,columns=DIRECTION_COLUMNS))


# Use the complete page viewer at the end; this function is available for manual calls.


## 7. Complete text log — every detected box
This notebook has **no text-deletion stage**. Every original Magi text must survive character cleanup; the assertions below catch any unexpected loss/reordering. Unassociated text is logged and displayed rather than disappearing. Log raw/cleaned character links, entity IDs, text-tail links and the exact reason a direction test was not possible. Missing association does not mean deletion.

Exports: `all_text_boxes.json/csv`, `text_logging_summary.csv`, `character_cleanup_log.json`. Direction rows without a character link cannot have an angle; they still appear in this complete per-text audit.

In [17]:
def build_complete_text_log(raw_pages,cleaned_pages,entity_map,audit_rows):
    logs=[]
    for page_index,(raw,cleaned) in enumerate(zip(raw_pages,cleaned_pages)):
        assert raw['texts']==cleaned['texts'],'Text boxes changed during character cleanup'
        assert raw['ocr']==cleaned['ocr'],'OCR order changed during character cleanup'
        for text_index,box in enumerate(raw['texts']):
            raw_characters=sorted({int(c) for t,c in raw['text_character_associations'] if int(t)==text_index})
            characters=sorted({int(c) for t,c in cleaned['text_character_associations'] if int(t)==text_index})
            tails=sorted({int(tail) for t,tail in cleaned.get('text_tail_associations',[]) if int(t)==text_index})
            entity_ids=list(dict.fromkeys(entity_map[f'PG{page_index+1}:C{c+1}'] for c in characters))
            tests=[r for r in audit_rows if r['page_index']==page_index and r['text_index']==text_index
                and str(r.get('page_file',''))==str(PAGE_FILES[page_index])]
            assessed=[r for r in tests if r.get('assessed')]
            flagged=any(r.get('direction_flag',False) for r in assessed)
            if not characters:status='no_character_association';reason='Magi found text but did not link it to a character'
            elif not tails:status='no_tail_association';reason='No explicit text-to-tail link; direction angle unavailable'
            elif not tests:status='direction_audit_not_run';reason='Run the direction audit cell before this logger'
            elif not assessed:status='invalid_vector';reason='Tail/character center coincides with text center, or coordinates are invalid'
            elif flagged:status='direction_flagged';reason='At least one tail/character angle exceeds the threshold'
            else:status='direction_agrees';reason='All assessed tail/character angles are within the threshold'
            logs.append(dict(page_index=page_index,page_file=str(PAGE_FILES[page_index]),
                box_id=f'PG{page_index+1}:T{text_index+1}',text_index=text_index,text=raw['ocr'][text_index],box=box,
                deleted=False,deletion_reason=None,display_status='available_in_all_texts_view',
                association_status='unassociated' if not characters else 'linked' if len(entity_ids)==1 else 'multiple_entities',
                raw_character_indices=raw_characters,cleaned_character_indices=characters,entity_ids=entity_ids,
                tail_indices=tails,has_character=bool(characters),has_tail=bool(tails),
                direction_status=status,direction_reason=reason,direction_assessed=bool(assessed),direction_flag=flagged,
                max_angle_degrees=max((r['angle_degrees'] for r in assessed),default=None)))
    assert len(logs)==sum(len(p['texts']) for p in raw_pages),'Missing text audit records'
    return logs


all_text_records=build_complete_text_log(magi_raw_results,character_results,entities['char_to_entity'],
    globals().get('direction_records',[]))
all_text_log=pd.DataFrame(all_text_records,columns=['page_index','page_file','box_id','text_index','text','box',
    'deleted','deletion_reason','display_status','association_status','raw_character_indices',
    'cleaned_character_indices','entity_ids','tail_indices','has_character','has_tail',
    'direction_status','direction_reason','direction_assessed','direction_flag','max_angle_degrees'])
summary=[]
for page_index,path in enumerate(PAGE_FILES):
    records=[r for r in all_text_records if r['page_index']==page_index]
    summary.append(dict(page=path.name,detected_texts=len(magi_raw_results[page_index]['texts']),
        logged_texts=len(records),deleted_texts=0,without_character=sum(not r['has_character'] for r in records),
        without_tail=sum(not r['has_tail'] for r in records),direction_assessed=sum(r['direction_assessed'] for r in records),
        direction_flagged=sum(r['direction_flag'] for r in records),
        character_detections=len(magi_raw_results[page_index]['characters']),
        characters_retained=len(character_results[page_index]['characters'])))
text_logging_summary=pd.DataFrame(summary)
display(text_logging_summary)
display(all_text_log[['box_id','text','association_status','entity_ids','tail_indices','deleted','direction_status','direction_reason']])
write_json(OUTPUT/'all_text_boxes.json',all_text_records)
all_text_log.to_csv(OUTPUT/'all_text_boxes.csv',index=False)
text_logging_summary.to_csv(OUTPUT/'text_logging_summary.csv',index=False)
write_json(OUTPUT/'character_cleanup_log.json',cleanup['audit'])
print('No text deletions are performed. Missing links are retained and shown in orange.')


,page,detected_texts,logged_texts,deleted_texts,without_character,without_tail,direction_assessed,direction_flagged,character_detections,characters_retained
0,01.png,11,11,0,2,2,7,0,7,7
1,02.png,9,9,0,0,1,8,0,7,7
2,03.png,8,8,0,0,2,6,0,8,8


,box_id,text,association_status,entity_ids,tail_indices,deleted,direction_status,direction_reason
0,PG1:T1,"MANGA STREAN, DO NOTMIROR",linked,[E3],[],False,no_tail_association,No explicit text-to-tail link; direction angle...
1,PG1:T2,Got your tomato devil right here.,linked,[E3],[3],False,direction_agrees,All assessed tail/character angles are within ...
2,PG1:T3,"It revolves from its seeds, so you should burn...",linked,[E3],[0],False,direction_agrees,All assessed tail/character angles are within ...
3,PG1:T4,"Good work, Denji.",linked,[E1],[4],False,direction_agrees,All assessed tail/character angles are within ...
4,PG1:T5,This guy's corpse'll fetch a pretty penny on t...,linked,[E1],[4],False,direction_agrees,All assessed tail/character angles are within ...
5,PG1:T6,The reward's 400 grand.,linked,[E1],[1],False,direction_agrees,All assessed tail/character angles are within ...
6,PG1:T7,Though you look at the bottom-of-the-the-the-a...,linked,[E1],[],False,no_tail_association,No explicit text-to-tail link; direction angle...
7,PG1:T8,Thanks !!,linked,[E3],[5],False,direction_agrees,All assessed tail/character angles are within ...
8,PG1:T9,I'm taking out 170 grand to cover your loan an...,unassociated,[],[6],False,no_character_association,Magi found text but did not link it to a chara...
9,PG1:T10,And then there's the commission for referring ...,unassociated,[],[7],False,no_character_association,Magi found text but did not link it to a chara...


No text deletions are performed. Missing links are retained and shown in orange.


## 8. Complete page viewer — all text by default
Text, OCR, IDs, characters, tails, panels, same-entity lines, original text-character links, text-tail links, direction arrows and removed character detections are individually selectable. Orange text has no character association; red text has a flagged direction. Entity colors are consistent across pages. **All** displays every detected text, including empty OCR and missing associations. The table below the page explains each text's status. Filters are explicit; their visible/total count is displayed. No model inference or prediction changes occur here.

In [18]:
def show_complete_page(page_index=0,text_filter='All',texts=True,characters=True,tails=False,
    panels=False,ocr=True,ids=True,same_entity=False,text_character=True,text_tail=True,
    direction_vectors=False,removed_characters=False):
    result=character_results[page_index];raw=magi_raw_results[page_index]
    records=[r for r in all_text_records if r['page_index']==page_index]
    def included(row):
        if text_filter=='No character link':return not row['has_character']
        if text_filter=='Direction flagged':return row['direction_flag']
        if text_filter=='Direction unassessed':return not row['direction_assessed']
        if text_filter=='Empty OCR':return not str(row['text'] or '').strip()
        return True
    visible=[r for r in records if included(r)];visible_indices={r['text_index'] for r in visible}
    palette=plt.get_cmap('hsv',max(2,len(entities['entities'])+1))
    colors={eid:palette(i) for i,eid in enumerate(entities['entities'])}
    fig,ax=plt.subplots(figsize=(12,18));ax.imshow(crop_pages[page_index])
    def midpoint(box):a,b,c,d=box;return ((a+c)/2,(b+d)/2)
    def mark(box,color,label='',style='-'):
        a,b,c,d=box;ax.add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,edgecolor=color,linestyle=style,linewidth=2))
        if label:ax.text(a,b,label,fontsize=8,bbox=dict(facecolor=color,alpha=.8))
    if panels:
        for i,box in enumerate(result.get('panels',[])):mark(box,'lime',f'PN{i+1}' if ids else '')
    if texts:
        for row in visible:
            color='red' if row['direction_flag'] else 'darkorange' if not row['has_character'] else (
                colors[row['entity_ids'][0]] if len(row['entity_ids'])==1 else 'gold')
            parts=[row['box_id']] if ids else []
            parts.append('/'.join(row['entity_ids']) if row['entity_ids'] else 'NO CHARACTER LINK')
            if row['direction_flag']:parts.append(f"ANGLE {row['max_angle_degrees']:.1f} deg")
            if ocr:parts.append(str(row['text'] or '<EMPTY OCR>')[:100])
            mark(row['box'],color,' | '.join(parts))
    members={}
    for i,box in enumerate(result['characters']):
        cid=f'PG{page_index+1}:C{i+1}';eid=entities['char_to_entity'][cid]
        members.setdefault(eid,[]).append(i)
        if characters:mark(box,colors[eid],(cid+' / ' if ids else '')+eid)
    if removed_characters:
        audit=cleanup['audit'][page_index]
        retained=set(audit['kept_original_indices'])
        for i,box in enumerate(raw['characters']):
            if i not in retained:
                target=audit['original_to_cleaned_index'][i]
                mark(box,'grey',f'REMOVED original C{i+1} -> C{target+1}' if ids else 'REMOVED CHARACTER','--')
    if tails:
        for i,box in enumerate(result.get('tails',[])):mark(box,'magenta',f'TAIL{i+1}' if ids else '')
    if same_entity:
        for eid,indices in members.items():
            origin=midpoint(result['characters'][indices[0]])
            for i in indices[1:]:
                point=midpoint(result['characters'][i]);ax.plot([origin[0],point[0]],[origin[1],point[1]],color=colors[eid])
    for enabled,key,target,style in [(text_character,'text_character_associations','characters','--'),
                                   (text_tail,'text_tail_associations','tails',':')]:
        if not enabled:continue
        for text_index,target_index in result.get(key,[]):
            if text_index not in visible_indices:continue
            origin=midpoint(result['texts'][text_index]);point=midpoint(result[target][target_index])
            color='magenta' if target=='tails' else colors[entities['char_to_entity'][f'PG{page_index+1}:C{target_index+1}']]
            ax.plot([origin[0],point[0]],[origin[1],point[1]],color=color,linestyle=style,linewidth=1.5)
    if direction_vectors:
        for row in globals().get('direction_records',[]):
            if row['page_index']!=page_index or row['text_index'] not in visible_indices or not row.get('assessed'):continue
            if str(row.get('page_file',''))!=str(PAGE_FILES[page_index]):continue
            origin=row['text_center']
            for point,color in [(row['tail_center'],'magenta'),(row['character_center'],'red' if row['direction_flag'] else 'cyan')]:
                ax.annotate('',xy=point,xytext=origin,arrowprops=dict(arrowstyle='->',color=color,lw=2))
    ax.set_title(f'{SEQUENCE_ID} | {PAGE_FILES[page_index].name} | visible texts {len(visible)}/{len(records)} | deleted 0 | filter: {text_filter}')
    ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))
    display(pd.DataFrame(visible,columns=all_text_log.columns)[['box_id','text','association_status','entity_ids',
        'tail_indices','deleted','direction_status','max_angle_degrees','direction_reason']])


import ipywidgets as widgets
# Hide previous partial viewers from this kernel; this is the complete replacement.
for name in ['_entity_viewer','_direction_viewer']:
    old=globals().get(name)
    if old:
        for item in old[:3]:item.unobserve_all()
        for item in old:item.close()
old=globals().get('_complete_entity_viewer')
if old:
    for item in old['controls'].values():item.unobserve_all();item.close()
    old['output'].close();old['panel'].close()
clear_output(wait=True)
complete_page=widgets.Dropdown(options=[(p.name,i) for i,p in enumerate(PAGE_FILES)],description='Page')
complete_filter=widgets.Dropdown(options=['All','No character link','Direction flagged','Direction unassessed','Empty OCR'],description='Texts')
flags={key:widgets.Checkbox(value=value,description=key.replace('_',' '),indent=False)
    for key,value in dict(texts=True,characters=True,tails=False,panels=False,ocr=True,ids=True,
        same_entity=False,text_character=True,text_tail=True,direction_vectors=False,removed_characters=False).items()}
controls=dict(page_index=complete_page,text_filter=complete_filter,**flags)
complete_output=widgets.Output()
def refresh_complete(change=None):
    with complete_output:
        clear_output(wait=True);show_complete_page(**{key:item.value for key,item in controls.items()})
for item in controls.values():item.observe(refresh_complete,names='value')
items=list(flags.values())
layout=[widgets.HBox([complete_page,complete_filter])]+[widgets.HBox(items[i:i+4]) for i in range(0,len(items),4)]
complete_panel=widgets.VBox(layout+[complete_output])
_complete_entity_viewer=dict(controls=controls,output=complete_output,panel=complete_panel)
display(complete_panel);refresh_complete()
